# Swept Pazy wing: continuous turbulence WRBM load sensitivity to structural mass and stiffness uncertainty

Obtain the sensitivity of the RMS WRBM amplitude when subject to continuous-turbulence. Use this sensitivity to propogate uncertainty, to find the uncertainty in this property due to uncertainty in the structural mass and stiffness properties.

## Imports

In [ ]:
import jax
import numpy as np
from jax import Array
from jax import numpy as jnp
from matplotlib import pyplot as plt

from flapjax.aero.flowfields import ConstantFlowField
from flapjax.aero.frequency_flowfields import VonKarmanFlowField
from flapjax.aero.gradients.data_structures import AeroGradsToCompute
from flapjax.coupled import (
    AeroelasticDesignVariables,
    AeroelasticFullStates,
    AeroelasticGradsToCompute,
    AeroelasticOutputUnflattened,
    CoupledAeroelastic,
    compute_gust_frf,
    gust_frf_adjoint,
)
from flapjax.models.pazy.base import (
    build_lumped_mass_matrices,
    build_stiffness_matrices,
    interp_cs_property,
)
from flapjax.models.pazy.swept.data.properties_10_deg import SWEEP_10_LE_CORRECTED
from flapjax.models.pazy.swept.swept_pazy_wing import generate_swept_pazy_wing
from flapjax.structure.gradients.data_structures import StructureGradsToCompute
from flapjax.structure.utils import get_solve_dofs
from flapjax.utils.print_utils import set_verbosity

set_verbosity("silent")  # remove flapjax console prints given the large number of solves below

## Discretisation, wing setup, and flight condition

Same swept Pazy wing / mass / stiffness model as `flutter_uncertainty.ipynb` (10 degree sweep,
leading-edge frequency-corrected tip mass, 16 lumped-mass keypoints, `node_multiplier=1`).

`U_INF` is fixed at 35 m/s, below the flutter onset.

In [ ]:
M = 8  # wing chordwise discretisation
M_STAR = 40  # number of wake panels per strip
AOA = jnp.deg2rad(5.0)  # angle of attack
RHO = 1.225
U_INF = 35.0  # m/s
DATA = SWEEP_10_LE_CORRECTED  # mass/CG/inertia/stiffness data
N_MASS = DATA.mass.shape[0]
N_ELEM = N_MASS - 1
ROOT_ELEM = 0  # index of the root element
WRBM_COMPONENT = 4  # wing root bending moment is given as: f_elem[0, 4]

MASS0 = DATA.mass  # (N_MASS,)
CG0 = jnp.stack((DATA.cg_x, DATA.cg_y, DATA.cg_z), axis=1)  # (N_MASS, 3)
INERTIA0 = jnp.stack((DATA.i_xx, DATA.i_yy, DATA.i_zz, DATA.i_xy, DATA.i_xz, DATA.i_yz), axis=1)  # (N_MASS, 6)
CG_FIELDS = ("cg_x", "cg_y", "cg_z")
INERTIA_FIELDS = ("i_xx", "i_yy", "i_zz", "i_xy", "i_xz", "i_yz")
STIFFNESS_FIELDS = ("k11", "k22", "k33", "k44", "k12", "k13", "k14", "k23", "k24", "k34")

CG_LABELS = ("$r_x$", "$r_y$", "$r_z$")
INERTIA_LABELS = ("$I_{xx}$", "$I_{yy}$", "$I_{zz}$", "$I_{xy}$", "$I_{xz}$", "$I_{yz}$")

M_LUMPED0 = build_lumped_mass_matrices(DATA, mass=MASS0, cg=CG0, inertia=INERTIA0)

seg_lengths = jnp.linalg.norm(jnp.diff(DATA.coords, axis=0), axis=1)
s_data = jnp.concatenate([jnp.zeros(1), jnp.cumsum(seg_lengths)])
ELEM_MIDPOINTS = 0.5 * (s_data[:-1] + s_data[1:])
K_CS_PARAMS: dict[str, Array] = {name: getattr(DATA, name) for name in STIFFNESS_FIELDS}  # each (N_ELEM,)


def k_cs_of_params(
        k11: Array, k22: Array, k33: Array, k44: Array,
        k12: Array, k13: Array, k14: Array, k23: Array, k24: Array, k34: Array,
) -> Array:
    k_cs_data = build_stiffness_matrices(
        DATA, k11=k11, k22=k22, k33=k33, k44=k44, k12=k12, k13=k13, k14=k14, k23=k23, k24=k24, k34=k34
    )
    return interp_cs_property(data=k_cs_data, data_coords=ELEM_MIDPOINTS, coords=ELEM_MIDPOINTS)


K_CS0 = k_cs_of_params(**K_CS_PARAMS)

base_wing: CoupledAeroelastic = generate_swept_pazy_wing(
    m=M,
    m_star=M_STAR,
    sweep_angle=10,
    tip_mass="LE_CORRECTED",
    node_multiplier=1,
    aoa=AOA,
    lumped_mass=True,
    flowfield=ConstantFlowField(
        u_inf=jnp.array((U_INF, 0.0, 0.0)),
        rho=RHO,
        relative_motion=True,
    ),
)


def apply_dv(wing: CoupledAeroelastic, m_lumped: Array, k_cs: Array) -> None:
    """Set the design variables for a given lumped-mass matrix array and per-element stiffness
    matrix array."""
    wing.set_design_variables(
        coords=wing.structure.x0,
        k_cs=k_cs,
        m_cs=wing.structure.m_cs,
        m_lumped=m_lumped,
        dt=0.1 / (M * U_INF),
        flowfield=ConstantFlowField(u_inf=jnp.array([U_INF, 0.0, 0.0]), rho=RHO, relative_motion=True),
        zeta_b0=wing.aero.zeta_b0,
        remove_checks=True,
    )


apply_dv(base_wing, M_LUMPED0, K_CS0)
case0 = base_wing.static_solve(prescribed_dofs=tuple(range(6)), horseshoe=False)
VARPHI_EQ = case0.structure.varphi

## Continuous turbulence spectrum

A von Karman vertical-gust PSD convected past the wing with frozen-turbulence hypothesis.

In [ ]:
SIGMA_GUST = 2.0  # m/s, RMS vertical gust velocity
LENGTH_SCALE_GUST = 1.0  # m
FRF_FLOWFIELD = VonKarmanFlowField(sigma=SIGMA_GUST, length_scale=LENGTH_SCALE_GUST, u_inf=U_INF)

## WRBM readout vector

`compute_gust_frf` returns the FRF of the free structural DOFs (`q`, `q_dot`). The WRBM comes from `f_elem = k_cs @ eps`, linear in `k_cs`, so
`D_EPS0_D_VARPHI_FREE` — the geometric Jacobian of the root element's strain w.r.t. the free DOFs,
frozen at the reference equilibrium `VARPHI_EQ` — lets the (perturbed) stiffness row be applied
after a fixed geometric factor, analogous to the flutter notebook freezing its tracked critical
eigenvalue.

In [ ]:
N_DOF = base_wing.structure.n_dof
SOLVE_DOFS = jnp.array(get_solve_dofs(n_dof=N_DOF, prescribed_dofs=case0.structure.prescribed_dofs))


def _eps_root_of_varphi(varphi_flat: Array) -> Array:
    hg = base_wing.structure.compute_hg_from_varphi(varphi=varphi_flat.reshape(-1, 6))
    d = base_wing.structure.make_d(hg)
    return base_wing.structure.make_eps(d=d)[ROOT_ELEM, :]  # (6,)


D_EPS0_D_VARPHI_FREE = jax.jacrev(_eps_root_of_varphi)(VARPHI_EQ.ravel())[:, SOLVE_DOFS]  # (6, n_free_dof)


def c_wrbm_of_k_cs(k_cs: Array) -> Array:
    return k_cs[ROOT_ELEM, WRBM_COMPONENT, :] @ D_EPS0_D_VARPHI_FREE  # (n_free_dof,)


C_WRBM0 = c_wrbm_of_k_cs(K_CS0)

## Nominal WRBM response spectrum, and its RMS value

`h_gust.q` is already PSD-weighted (`LinearCoupled.frf_base`), so `|h_gust.q(omega)|**2` is
directly the output PSD and `Var(WRBM) = integral_0^inf |h_wrbm(omega)|**2 d(omega)`, approximated
below by trapezoidal quadrature over `OMEGA`, with `A_rms = sqrt(Var(WRBM))`. This RMS/PSD-integral
is a standard continuous-turbulence certification metric.

In [ ]:
def wrbm_rms(h_wrbm: Array, omega: Array) -> Array:
    """RMS of `h_wrbm`, via trapezoidal quadrature over `omega`."""
    return jnp.sqrt(jnp.trapezoid(jnp.abs(h_wrbm) ** 2, omega))


OMEGA_MAX = 2.0 * jnp.pi * 50.0  # rad/s, 0-30 Hz
dv0 = base_wing.get_design_variables(case=case0, grads_to_compute=None)

N_OMEGA = 200
OMEGA = jnp.linspace(0.0, OMEGA_MAX, N_OMEGA)

h0 = compute_gust_frf(base_wing, dv0, VARPHI_EQ, case0, OMEGA, FRF_FLOWFIELD)
h_wrbm0 = h0.q @ C_WRBM0  # (n_freq,) complex
a_vec0 = jnp.abs(h_wrbm0)
a_rms0 = float(wrbm_rms(h_wrbm0, OMEGA))
print(f"\nRMS WRBM response: A_rms = {a_rms0:.4g}")

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.plot(np.array(OMEGA) / (2.0 * np.pi), np.array(a_vec0), color="tab:blue", label="WRBM response magnitude")
ax.set_xlabel("Frequency [Hz]")
ax.set_ylabel(r"WRBM response magnitude, $|H_\mathrm{WRBM}(\omega)|$")
ax.set_title("Continuous-turbulence WRBM response spectrum")
plt.tight_layout()
plt.show()

## Adjoint sensitivity of the RMS WRBM response to mass and stiffness

`gust_frf_adjoint` with `omega=OMEGA` (the whole swept band) gives the gradient of
`A_rms = wrbm_rms(H_WRBM(OMEGA), OMEGA)` with respect to every lumped mass and stiffness matrix
in one adjoint solve, batched over the whole band rather than a single frequency.

In [ ]:
def wrbm_objective(
        states: AeroelasticFullStates, dv: AeroelasticDesignVariables, h_gust: AeroelasticOutputUnflattened,
) -> Array:
    """RMS WRBM frequency response across the whole sampled spectrum."""
    del states
    k_cs = dv.structure.k_cs
    assert k_cs is not None
    c_wrbm = c_wrbm_of_k_cs(k_cs)
    h_wrbm = h_gust.q @ c_wrbm  # (n_freq,) complex
    return wrbm_rms(h_wrbm, OMEGA)


grads_to_compute = AeroelasticGradsToCompute(
    structure=StructureGradsToCompute(k_cs=True, m_cs=False, m_lumped=True),
    aero=AeroGradsToCompute(zeta_b0=False, flowfield=False),
)

a_rms_adj, dv_bar = gust_frf_adjoint(
    system=base_wing, case=case0, omega=OMEGA, objective=wrbm_objective,
    frf_flowfield=FRF_FLOWFIELD, grads_to_compute=grads_to_compute,
)
print(f"A_rms (adjoint primal) = {float(a_rms_adj):.4g}")

dA_dm_lumped = dv_bar.structure.m_lumped  # (N_MASS, 6, 6)
dA_dk_cs = dv_bar.structure.k_cs  # (N_ELEM, 6, 6)

## Pulling the gradient back to mass, CG, inertia, and stiffness

Same `jax.vjp` pullback through `build_lumped_mass_matrices` and `k_cs_of_params` as the
flutter notebook, reusing the same adjoint result.

In [ ]:
def m_lumped_of_params(mass: Array, cg: Array, inertia: Array) -> Array:
    return build_lumped_mass_matrices(DATA, mass=mass, cg=cg, inertia=inertia)


_, vjp_mass_params = jax.vjp(m_lumped_of_params, MASS0, CG0, INERTIA0)
dA_dmass, dA_dcg, dA_dinertia = vjp_mass_params(dA_dm_lumped)

_, vjp_stiffness_params = jax.vjp(k_cs_of_params, *(K_CS_PARAMS[name] for name in STIFFNESS_FIELDS))
dA_dstiffness = dict(zip(STIFFNESS_FIELDS, vjp_stiffness_params(dA_dk_cs)))

## Validation against finite differences

Full nonlinear re-solve (static solve -> `compute_gust_frf` -> `wrbm_rms`) for the most impactful
mass and stiffness entries, compared against the adjoint-derived slope. The solve is jitted once
and reused for every FD check and Monte Carlo sample below.

In [ ]:
def _static_solve_and_wrbm_response(wing: CoupledAeroelastic, m_lumped: Array, k_cs: Array) -> Array:
    apply_dv(wing, m_lumped, k_cs)
    static_sol = wing.static_solve(prescribed_dofs=tuple(range(6)), horseshoe=False)
    dv = wing.get_design_variables(case=static_sol, grads_to_compute=None)
    h = compute_gust_frf(wing, dv, static_sol.structure.varphi, static_sol, OMEGA, FRF_FLOWFIELD)
    h_wrbm = h.q @ c_wrbm_of_k_cs(k_cs)
    return wrbm_rms(h_wrbm, OMEGA)


_wrbm_response_jit = jax.jit(_static_solve_and_wrbm_response)


def wrbm_response_pipeline(m_lumped: Array, k_cs: Array) -> float:
    return float(_wrbm_response_jit(base_wing, m_lumped, k_cs))


idx_mass = int(jnp.argmax(jnp.abs(dA_dmass)))
eps_mass = 0.05
delta_mass = float(MASS0[idx_mass]) * eps_mass
m_lumped_plus = build_lumped_mass_matrices(DATA, mass=MASS0.at[idx_mass].add(delta_mass), cg=CG0, inertia=INERTIA0)
m_lumped_minus = build_lumped_mass_matrices(DATA, mass=MASS0.at[idx_mass].add(-delta_mass), cg=CG0, inertia=INERTIA0)
a_plus = wrbm_response_pipeline(m_lumped_plus, K_CS0)
a_minus = wrbm_response_pipeline(m_lumped_minus, K_CS0)
fd_grad_mass = (a_plus - a_minus) / (2.0 * delta_mass)
adj_grad_mass = float(dA_dmass[idx_mass])

### Most impactful mass and stiffness entries

Repeats the finite-difference validation above, now for the single most impactful stiffness
entry (largest-magnitude adjoint sensitivity across `STIFFNESS_UQ_FIELDS` and all elements).

In [ ]:
STIFFNESS_UQ_FIELDS = ("k22", "k33", "k44")
STIFFNESS_UQ_LABELS = ("$GJ$", "$EIy$", "$EIz$")  # torsional, out-of-plane, in-plane

best_name, best_idx, best_grad = None, None, 0.0
for name in STIFFNESS_UQ_FIELDS:
    grad_arr = np.array(jnp.abs(dA_dstiffness[name]))
    idx = int(np.argmax(grad_arr))
    grad = float(dA_dstiffness[name][idx])
    if abs(grad) > abs(best_grad):
        best_name, best_idx, best_grad = name, idx, grad

eps_stiff = 0.05
delta_stiff = float(K_CS_PARAMS[best_name][best_idx]) * eps_stiff
stiffness_plus = dict(K_CS_PARAMS)
stiffness_plus[best_name] = K_CS_PARAMS[best_name].at[best_idx].add(delta_stiff)
stiffness_minus = dict(K_CS_PARAMS)
stiffness_minus[best_name] = K_CS_PARAMS[best_name].at[best_idx].add(-delta_stiff)

a_plus_stiff = wrbm_response_pipeline(M_LUMPED0, k_cs_of_params(**stiffness_plus))
a_minus_stiff = wrbm_response_pipeline(M_LUMPED0, k_cs_of_params(**stiffness_minus))
fd_grad_stiff = (a_plus_stiff - a_minus_stiff) / (2.0 * delta_stiff)
print(f"Most impactful stiffness parameter: element {best_idx} {best_name}")
print(f"  adjoint dA/d(.)={best_grad:.4g}, FD={fd_grad_stiff:.4g}, "
      f"relative difference={abs(fd_grad_stiff - best_grad) / abs(fd_grad_stiff):.2%}")

## Assumed per-parameter uncertainty

Same `UNCERTAINTY_SPEC` convention as the flutter notebook: every variable is an independent
Gaussian, given as a coefficient of variation (`"cov"`) or an absolute value (`"abs"`, for CG
offsets). `k11`/`k14` are excluded from `STIFFNESS_UQ_FIELDS` entirely, matching the flutter
notebook's `STIFFNESS_FIELDS` scope.

In [ ]:
VARIABLES: dict[str, Array] = {
    "mass": MASS0,
    "cg_x": CG0[:, 0],
    "cg_y": CG0[:, 1],
    "cg_z": CG0[:, 2],
    "i_xx": INERTIA0[:, 0],
    "i_yy": INERTIA0[:, 1],
    "i_zz": INERTIA0[:, 2],
    "i_xy": INERTIA0[:, 3],
    "i_xz": INERTIA0[:, 4],
    "i_yz": INERTIA0[:, 5],
    **{name: K_CS_PARAMS[name] for name in STIFFNESS_UQ_FIELDS},
}

SENSITIVITIES: dict[str, Array] = {
    "mass": dA_dmass,
    "cg_x": dA_dcg[:, 0],
    "cg_y": dA_dcg[:, 1],
    "cg_z": dA_dcg[:, 2],
    "i_xx": dA_dinertia[:, 0],
    "i_yy": dA_dinertia[:, 1],
    "i_zz": dA_dinertia[:, 2],
    "i_xy": dA_dinertia[:, 3],
    "i_xz": dA_dinertia[:, 4],
    "i_yz": dA_dinertia[:, 5],
    **{name: dA_dstiffness[name] for name in STIFFNESS_UQ_FIELDS},
}

UNCERTAINTY_SPEC: dict[str, tuple[str, float]] = {
    "mass": ("cov", 0.02),
    "cg_x": ("abs", 0.5e-3),
    "cg_y": ("abs", 0.5e-3),
    "cg_z": ("abs", 0.5e-3),
    "i_xx": ("cov", 0.05),
    "i_yy": ("cov", 0.05),
    "i_zz": ("cov", 0.05),
    "i_xy": ("cov", 0.05),
    "i_xz": ("cov", 0.05),
    "i_yz": ("cov", 0.05),
    "k22": ("cov", 0.05),
    "k33": ("cov", 0.05),
    "k44": ("cov", 0.05),
}


def sigma_of(name: str, nominal: Array) -> Array:
    """Standard deviation for every entry of `nominal`."""
    mode, value = UNCERTAINTY_SPEC[name]
    if mode == "cov":
        return jnp.full_like(nominal, value * jnp.mean(jnp.abs(nominal)))
    if mode == "abs":
        return jnp.full_like(nominal, value)
    raise ValueError(f"Unknown uncertainty mode {mode!r} for variable {name!r}")


sigma = {name: sigma_of(name, nominal) for name, nominal in VARIABLES.items()}
var_contrib = {name: (SENSITIVITIES[name] * sigma[name]) ** 2 for name in VARIABLES}

a_rms0 = float(a_rms_adj)
var_a_rms = float(sum(jnp.sum(v) for v in var_contrib.values()))
std_a_rms = float(jnp.sqrt(var_a_rms))

CATEGORIES: dict[str, tuple[str, ...]] = {
    "mass": ("mass",),
    "cg": CG_FIELDS,
    "inertia": INERTIA_FIELDS,
    "stiffness": STIFFNESS_UQ_FIELDS,
}

## Sensitivity ranking

In [ ]:
entries = []
for i in range(N_MASS):
    entries.append((f"Node {i} mass", float(jnp.sqrt(var_contrib["mass"][i]))))
    for field, label in zip(CG_FIELDS, CG_LABELS):
        entries.append((f"Node {i} {label}", float(jnp.sqrt(var_contrib[field][i]))))
    for field, label in zip(INERTIA_FIELDS, INERTIA_LABELS):
        entries.append((f"Node {i} {label}", float(jnp.sqrt(var_contrib[field][i]))))
for i in range(N_ELEM):
    for field, label in zip(STIFFNESS_UQ_FIELDS, STIFFNESS_UQ_LABELS):
        entries.append((f"Element {i}, {label}", float(jnp.sqrt(var_contrib[field][i]))))

entries.sort(key=lambda e: e[1], reverse=True)
top_n = 15
labels_top = [e[0] for e in entries[:top_n]][::-1]
values_top = [e[1] for e in entries[:top_n]][::-1]

fig, ax = plt.subplots(figsize=(6, 5.5))
ax.barh(labels_top, values_top, color="tab:blue")
ax.set_xlabel(r"$A_\mathrm{rms}$ shift from this entry's 1-sigma uncertainty")
ax.set_title(f"Top {top_n} contributors to RMS WRBM response uncertainty")
plt.tight_layout()
plt.show()

### Spanwise sensitivity breakdown

Plot of sensitivity for each structural design parameter.

In [ ]:
y_span = np.array(DATA.coords[:, 1])
y_span_elem = np.array(0.5 * (DATA.coords[:-1, 1] + DATA.coords[1:, 1]))
tab10_colors = plt.get_cmap("tab10").colors

fig, ((ax_mass, ax_cg), (ax_inertia, ax_stiffness)) = plt.subplots(2, 2, figsize=(10, 8), sharex=True)

ax_mass.plot(y_span, dA_dmass, marker="o", color="tab:blue")
ax_mass.set_title("Mass")
ax_mass.set_ylabel(r"$\partial~A_\mathrm{rms}/\partial~m$")

for i, (label, color) in enumerate(zip(CG_LABELS, ("tab:blue", "tab:orange", "tab:green"))):
    ax_cg.plot(y_span, dA_dcg[:, i], marker="o", color=color, label=label)
ax_cg.set_title("Mass Offset")
ax_cg.set_ylabel(r"$\partial~A_\mathrm{rms}/\partial~\mathbf{r}$")
ax_cg.legend(fontsize=8)

for i, (label, color) in enumerate(zip(INERTIA_LABELS, tab10_colors[:6])):
    ax_inertia.plot(y_span, dA_dinertia[:, i], marker="o", color=color, label=label)
ax_inertia.set_title("Rotational Inertia")
ax_inertia.set_ylabel(r"$\partial~A_\mathrm{rms}/\partial~\mathbf{I}$")
ax_inertia.legend(fontsize=8, ncol=2)

for i, (field, label) in enumerate(zip(STIFFNESS_UQ_FIELDS, STIFFNESS_UQ_LABELS)):
    ax_stiffness.plot(y_span_elem, dA_dstiffness[field], marker="o", color=tab10_colors[i], label=label)
ax_stiffness.set_title("Stiffness")
ax_stiffness.set_ylabel(r"$\partial~A_\mathrm{rms}/\partial~\mathbf{K}_{cs}$")
ax_stiffness.legend(fontsize=7, ncol=2)

for ax in (ax_mass, ax_cg, ax_inertia, ax_stiffness):
    ax.axhline(0.0, color="k", linewidth=0.5)
for ax in (ax_inertia, ax_stiffness):
    ax.set_xlabel("Spanwise position, y [m]")

fig.suptitle(r"RMS WRBM response amplitude $A_\mathrm{rms}$ sensitivity to each parameter")
plt.tight_layout()
plt.show()

## Monte Carlo cross-check

Full nonlinear re-solves with every mass, CG, inertia, and stiffness entry perturbed
simultaneously, re-evaluating the RMS/PSD integral over the whole `OMEGA` band.

In [ ]:
N_MC = 200
rng = np.random.default_rng(0)


def sample_variable(name: str) -> Array:
    nominal = VARIABLES[name]
    noise = rng.standard_normal(nominal.shape)
    return nominal + sigma[name] * noise


mc_samples = []
for i_mc in range(N_MC):
    mass_sample = sample_variable("mass")
    cg_sample = jnp.stack([sample_variable(name) for name in CG_FIELDS], axis=1)
    inertia_sample = jnp.stack([sample_variable(name) for name in INERTIA_FIELDS], axis=1)
    m_lumped_sample = build_lumped_mass_matrices(DATA, mass=mass_sample, cg=cg_sample, inertia=inertia_sample)

    stiffness_sample = dict(K_CS_PARAMS)  # coupling terms excluded from UQ stay fixed at nominal
    stiffness_sample.update({name: sample_variable(name) for name in STIFFNESS_UQ_FIELDS})
    k_cs_sample = k_cs_of_params(**stiffness_sample)

    a_mc = wrbm_response_pipeline(m_lumped_sample, k_cs_sample)
    mc_samples.append(a_mc)
    print(f"  sample {i_mc + 1:3d}/{N_MC}: A_rms = {a_mc:.4g}")

mc_samples = np.array(mc_samples)
mc_mean, mc_std = mc_samples.mean(), mc_samples.std(ddof=1)
print(f"\nMonte Carlo ({mc_samples.size} samples): mean={mc_mean:.4g}, std={mc_std:.4g}")
print(f"FOSM prediction:                    mean={a_rms0:.4g}, std={std_a_rms:.4g}")

In [ ]:
half_width = max(0.1 * a_rms0, 6.0 * std_a_rms)
a_axis = jnp.linspace(a_rms0 - half_width, a_rms0 + half_width, 200)
gaussian_pdf = jnp.exp(-0.5 * ((a_axis - a_rms0) / std_a_rms) ** 2) / (std_a_rms * jnp.sqrt(2.0 * jnp.pi))

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.hist(mc_samples, bins=max(5, N_MC // 8), density=True, color="tab:blue", alpha=0.6, label="Monte Carlo samples")
ax.plot(np.array(a_axis), np.array(gaussian_pdf), color="tab:red", label="FOSM Gaussian approximation")
ax.axvline(a_rms0, color="k", linewidth=0.8, linestyle="--", label=f"A_rms0 = {a_rms0:.3g}")
ax.set_xlabel("RMS WRBM response amplitude")
ax.set_ylabel("Probability density")
ax.set_title("RMS continuous-turbulence WRBM response under structural uncertainty")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()